In [4]:
import numpy as np

# -------------------------
# precision control
# -------------------------
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# ============================================================
# MQMQA G^ex toy verifier with ternary modifier cases:
#   case_map controls which branch is used for specific terms.
#
# Flip case_map values between: "gamma", "nu", "else"
# ============================================================

# -------------------------
# USER SWITCH (ONLY THING CHANGED)
# -------------------------
# case_map = {"AB_XX": "gamma", "AB_YY": "gamma"}
# case_map = {"AB_XX": "nu",    "AB_YY": "nu"}
case_map = {"AB_XX": "else",  "AB_YY": "else"}

# FD step sizes
H_LIST = [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]

# Exponents for the toy Δg and ternary modifier
p_exp = DT("1.0")
q_exp = DT("1.0")
r_exp = DT("3.0")   # (r_exp - 1) appears as exponent, so r_exp >= 2 is typical

# -------------------------
# utilities
# -------------------------
def delta(u, v):
    return DT("1.0") if u == v else DT("0.0")

def ln_derivs(z, z_p, z_pq):
    """Return (ln z)_{,p} vector and (ln z)_{,pq} matrix."""
    if z <= 0:
        raise ValueError("log argument <= 0")
    ln_p  = z_p / z
    ln_pq = z_pq / z - np.outer(z_p, z_p) / (z*z)
    return ln_p, ln_pq

def ratio_derivs(y, y_p, y_pq, d, d_p, d_pq):
    """
    w = y/d
    w_p  = (y_p d - y d_p)/d^2
    w_pq = (y_pq d - y_p d_q - y_q d_p - y d_pq)/d^2 + 2 y d_p d_q / d^3
    """
    w = y / d
    w_p = (y_p*d - y*d_p) / (d*d)
    w_pq = (y_pq*d - np.outer(y_p, d_p) - np.outer(d_p, y_p) - y*d_pq) / (d*d) \
           + 2.0*y*np.outer(d_p, d_p) / (d**3)
    return w, w_p, w_pq

# -------------------------
# build quadruplets (3 cations A,B,C; 2 anions X,Y)
# 6 cation pairs (AA,AB,AC,BB,BC,CC) × 3 anion pairs (XX,XY,YY) = 18 vars
# -------------------------
cations = ["A", "B", "C"]
anions  = ["X", "Y"]

cat_pairs = []
for ii, i in enumerate(cations):
    for j in cations[ii:]:
        cat_pairs.append((i, j))

an_pairs = [("X", "X"), ("X", "Y"), ("Y", "Y")]

quad = []
names = []
for (i, j) in cat_pairs:
    for (k, l) in an_pairs:
        quad.append((i, j, k, l))
        names.append(f"{i}{j}_{k}{l}")

idx = {nm: i for i, nm in enumerate(names)}
M = len(names)

# -------------------------
# weights for Y_{m/k} (two-index Y used in ternary modifier)
# Y_{m/k} = sum_r X_r * ((δ_im+δ_jm)(δ_xk+δ_yk))/4,  X_r = n_r / Nquad
# -------------------------
def w_vec(m, k):
    w = np.zeros(M, dtype=DT)
    for p, (i, j, x, y) in enumerate(quad):
        w[p] = ((delta(i, m) + delta(j, m)) * (delta(x, k) + delta(y, k))) / DT("4.0")
    return w

W = {(m, k): w_vec(m, k) for m in cations for k in anions}

def Y_and_derivs(n, m, k):
    """
    Y = (w·n)/N, with N=sum(n)
    Y_p  = (w_p - Y)/N
    Y_pq = -(Y_p + Y_q)/N
    """
    n = asDT(n)
    N = np.sum(n, dtype=DT)
    if N <= 0:
        raise ValueError("Nquad <= 0")

    w = W[(m, k)]
    A = np.dot(w, n)
    Y = A / N

    Y_p = (w - Y) / N
    Y_pq = -(Y_p[:, None] + Y_p[None, :]) / N
    return Y, Y_p, Y_pq

# -------------------------
# Toy coefficients for diagonal (kk = XX, YY) terms
# -------------------------
g = {}
for (i, j) in cat_pairs:
    for kk in ["XX", "YY"]:
        g[f"{i}{j}_{kk}"] = DT(str(1500.0 + 100.0 * ((hash(f"{i}{j}_{kk}") % 11) - 5)))

# -------------------------
# Δg model with case_map driving the ternary modifier branch
# -------------------------
def delta_g_with_cases(n, case_map):
    """
    dg nonzero only for anion-diagonal terms (.._XX and .._YY).
    For each such r = ij/kk with k in {X,Y}:

      base = a^p b^q / (a+b)^(p+q), with a=ξ_{ij/k}, b=ξ_{ji/k}

    ξ definitions (toy but structured):
      - For AB: include C in both ξ's (so ternary can matter)
          a = Y_{A/k} + Y_{C/k}
          b = Y_{B/k} + Y_{C/k}
      - Otherwise:
          a = Y_{i/k}
          b = Y_{j/k}

    Ternary modifier M applied ONLY if rname in case_map:
      "gamma": M = (Y_{C/k}/b) * (1 - Y_{j/k}/b)^(r-1)
      "nu":    M = (Y_{C/k}/a) * (1 - Y_{i/k}/a)^(r-1)
      "else":  M =  Y_{C/k}    * (1 - a - b)^(r-1)

    Derivatives via log-diff:
      dg_p  = dg * Λ_p
      dg_pq = dg * (Λ_p Λ_q + Λ_pq)
    """
    n = asDT(n)
    if np.any(n <= 0):
        raise ValueError("All n must be > 0 for this toy.")

    dg    = np.zeros(M, dtype=DT)
    dg_p  = np.zeros((M, M), dtype=DT)
    dg_pq = np.zeros((M, M, M), dtype=DT)

    # Precompute Y_{m/k} and derivatives for all m,k
    Y = {}
    Yp = {}
    Ypq = {}
    for m in cations:
        for k in anions:
            val, val_p, val_pq = Y_and_derivs(n, m, k)
            Y[(m, k)]   = val
            Yp[(m, k)]  = val_p
            Ypq[(m, k)] = val_pq

    def fill_one(rname, i, j, k):
        r  = idx[rname]
        gr = g[rname]

        Yi, Yi_p, Yi_pq = Y[(i, k)], Yp[(i, k)], Ypq[(i, k)]
        Yj, Yj_p, Yj_pq = Y[(j, k)], Yp[(j, k)], Ypq[(j, k)]
        Ym, Ym_p, Ym_pq = Y[("C", k)], Yp[("C", k)], Ypq[("C", k)]  # m = C

        # ξ definitions
        if (i, j) == ("A", "B"):
            a    = Yi + Ym
            b    = Yj + Ym
            a_p  = Yi_p  + Ym_p
            b_p  = Yj_p  + Ym_p
            a_pq = Yi_pq + Ym_pq
            b_pq = Yj_pq + Ym_pq
        else:
            a, a_p, a_pq = Yi, Yi_p, Yi_pq
            b, b_p, b_pq = Yj, Yj_p, Yj_pq

        s    = a + b
        s_p  = a_p + b_p
        s_pq = a_pq + b_pq

        if a <= 0 or b <= 0 or s <= 0:
            raise ValueError("Need a,b,s > 0 (pick n away from boundaries).")

        base = (a**p_exp) * (b**q_exp) / (s**(p_exp + q_exp))

        # ternary modifier branch
        case = case_map.get(rname, "none")
        Mval = DT("1.0")

        if case != "none":
            if case == "gamma":
                w, w_p, w_pq = ratio_derivs(Yj, Yj_p, Yj_pq, b, b_p, b_pq)  # w = Yj/b
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                if Ym <= 0 or t <= 0:
                    raise ValueError("gamma case needs Ym>0 and (1 - Yj/b)>0")
                Mval = (Ym / b) * (t**(r_exp - 1.0))

            elif case == "nu":
                w, w_p, w_pq = ratio_derivs(Yi, Yi_p, Yi_pq, a, a_p, a_pq)  # w = Yi/a
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                if Ym <= 0 or t <= 0:
                    raise ValueError("nu case needs Ym>0 and (1 - Yi/a)>0")
                Mval = (Ym / a) * (t**(r_exp - 1.0))

            elif case == "else":
                t    = DT("1.0") - a - b
                t_p  = -(a_p + b_p)
                t_pq = -(a_pq + b_pq)
                if Ym <= 0 or t <= 0:
                    raise ValueError("else case needs Ym>0 and (1 - a - b)>0")
                Mval = Ym * (t**(r_exp - 1.0))

            else:
                raise ValueError(f"Unknown case label: {case}")

        val   = gr * base * Mval
        dg[r] = val

        # ---- base log-derivatives ----
        ln_a_p, ln_a_pq = ln_derivs(a, a_p, a_pq)
        ln_b_p, ln_b_pq = ln_derivs(b, b_p, b_pq)
        ln_s_p, ln_s_pq = ln_derivs(s, s_p, s_pq)

        Lambda_p  = p_exp*ln_a_p + q_exp*ln_b_p - (p_exp+q_exp)*ln_s_p
        Lambda_pq = p_exp*ln_a_pq + q_exp*ln_b_pq - (p_exp+q_exp)*ln_s_pq

        # ---- modifier log-derivatives (added) ----
        if case != "none":
            ln_Ym_p, ln_Ym_pq = ln_derivs(Ym, Ym_p, Ym_pq)

            if case == "gamma":
                w, w_p, w_pq = ratio_derivs(Yj, Yj_p, Yj_pq, b, b_p, b_pq)
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)
                ln_b_p, ln_b_pq = ln_derivs(b, b_p, b_pq)

                Lambda_p  = Lambda_p  + ln_Ym_p  - ln_b_p  + (r_exp-1.0)*ln_t_p
                Lambda_pq = Lambda_pq + ln_Ym_pq - ln_b_pq + (r_exp-1.0)*ln_t_pq

            elif case == "nu":
                w, w_p, w_pq = ratio_derivs(Yi, Yi_p, Yi_pq, a, a_p, a_pq)
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)
                ln_a_p, ln_a_pq = ln_derivs(a, a_p, a_pq)

                Lambda_p  = Lambda_p  + ln_Ym_p  - ln_a_p  + (r_exp-1.0)*ln_t_p
                Lambda_pq = Lambda_pq + ln_Ym_pq - ln_a_pq + (r_exp-1.0)*ln_t_pq

            elif case == "else":
                t    = DT("1.0") - a - b
                t_p  = -(a_p + b_p)
                t_pq = -(a_pq + b_pq)
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)

                Lambda_p  = Lambda_p  + ln_Ym_p  + (r_exp-1.0)*ln_t_p
                Lambda_pq = Lambda_pq + ln_Ym_pq + (r_exp-1.0)*ln_t_pq

        # dg derivatives from Lambda
        dg_p[r, :]   = val * Lambda_p
        dg_pq[r, :, :] = val * (np.outer(Lambda_p, Lambda_p) + Lambda_pq)

    # Only fill kk = XX and YY
    for (i, j) in cat_pairs:
        for kk in ["XX", "YY"]:
            k = "X" if kk == "XX" else "Y"
            rname = f"{i}{j}_{kk}"
            fill_one(rname, i, j, k)

    return dg, dg_p, dg_pq

# -------------------------
# Eq.17-like prefactors P and Q
# -------------------------
def P_Q_and_derivs(n):
    """
    Toy Eq.17 prefactors with Z=1.

    - P_{ij/XX} = 0.5*n_{ij/XY}, P_{ij/YY} = 0.5*n_{ij/XY}
    - Q_{ii/an} = 0.5 * sum_{m!=i} n_{im/an}  (two neighbors since 3 cations)
    """
    n = asDT(n)

    P  = np.zeros(M, dtype=DT)
    Q  = np.zeros(M, dtype=DT)
    Pp = np.zeros((M, M), dtype=DT)
    Qp = np.zeros((M, M), dtype=DT)

    # P
    for (i, j) in cat_pairs:
        nm_xy = f"{i}{j}_XY"
        ixy = idx[nm_xy]
        for kk in ["XX", "YY"]:
            rnm = f"{i}{j}_{kk}"
            r = idx[rnm]
            P[r] = DT("0.5") * n[ixy]
            Pp[r, ixy] = DT("0.5")

    # Q
    for i in cations:
        for an in ["XX", "XY", "YY"]:
            rnm = f"{i}{i}_{an}"
            r = idx[rnm]
            s = DT("0.0")
            for m in cations:
                if m == i:
                    continue
                a, b = (i, m) if i <= m else (m, i)
                nm_im = f"{a}{b}_{an}"
                jidx = idx[nm_im]
                s += n[jidx]
                Qp[r, jidx] += DT("0.5")
            Q[r] = DT("0.5") * s

    return P, Q, Pp, Qp

# -------------------------
# G^ex and analytic Hessian
# -------------------------
def G_ex(n, case_map):
    n = asDT(n)
    dg, _, _ = delta_g_with_cases(n, case_map)
    P, Q, _, _ = P_Q_and_derivs(n)

    T1 = np.dot(n, dg)

    diag_l_eq_k = [idx[nm] for nm in names if (nm.endswith("_XX") or nm.endswith("_YY"))]
    T2 = np.dot(P[diag_l_eq_k], dg[diag_l_eq_k])

    diag_j_eq_i = [idx[nm] for nm in names if nm[0] == nm[1]]  # ii/..
    T3 = np.dot(Q[diag_j_eq_i], dg[diag_j_eq_i])

    return DT("0.5") * (T1 + T2 + T3)

def H_ex_analytic(n, case_map):
    n = asDT(n)
    dg, dg_p, dg_pq = delta_g_with_cases(n, case_map)
    P, Q, Pp, Qp = P_Q_and_derivs(n)

    H = np.zeros((M, M), dtype=DT)

    diag_l_eq_k = [idx[nm] for nm in names if (nm.endswith("_XX") or nm.endswith("_YY"))]
    diag_j_eq_i = [idx[nm] for nm in names if nm[0] == nm[1]]

    for p in range(M):
        for q in range(M):
            term = DT("0.0")

            # T1_pq
            term += dg_p[p, q] + dg_p[q, p]
            term += np.dot(n, dg_pq[:, p, q])

            # T2_pq
            for r in diag_l_eq_k:
                term += Pp[r, p]*dg_p[r, q] + Pp[r, q]*dg_p[r, p] + P[r]*dg_pq[r, p, q]

            # T3_pq
            for r in diag_j_eq_i:
                term += Qp[r, p]*dg_p[r, q] + Qp[r, q]*dg_p[r, p] + Q[r]*dg_pq[r, p, q]

            H[p, q] = DT("0.5") * term

    return H

# -------------------------
# FD Hessian on scalar G^ex
# -------------------------
def H_fd(n0, h, case_map):
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large: n0-h must remain positive for all components.")

    H = np.zeros((M, M), dtype=DT)
    G0 = G_ex(n0, case_map)

    # diagonal
    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p] = DT("1.0")
        H[p, p] = (G_ex(n0+h*e, case_map) - DT("2.0") * G0 + G_ex(n0-h*e, case_map)) / (h*h)

    # off-diagonal
    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p] = DT("1.0")
        for q in range(p+1, M):
            eq = np.zeros(M, dtype=DT); eq[q] = DT("1.0")
            Gpp = G_ex(n0+h*ep+h*eq, case_map)
            Gpm = G_ex(n0+h*ep-h*eq, case_map)
            Gmp = G_ex(n0-h*ep+h*eq, case_map)
            Gmm = G_ex(n0-h*ep-h*eq, case_map)
            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0")*h*h)
            H[p, q] = val
            H[q, p] = val

    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))

# -------------------------
# RUN
# -------------------------
if __name__ == "__main__":
    # Positive base point (18 vars). Keep away from boundaries so logs stay valid.
    n0 = np.array([DT(str(1.0 + 0.03*i)) for i in range(M)], dtype=DT)

    print("M =", M)
    print("case_map =", case_map)
    print("First 12 vars:", names[:12], "...\n")

    G0 = G_ex(n0, case_map)
    print("G_ex(n0) =", G0)

    Ha = H_ex_analytic(n0, case_map)
    print("||H_analytic||_F =", float(fro_norm(Ha)))
    print("analytic symmetry err =", float(fro_norm(Ha - Ha.T) / max(DT("1.0"), fro_norm(Ha))))

    for h in H_LIST:
        if np.min(n0) <= h:
            print(f"skip h={h:g} (would make some n negative)")
            continue
        Hfd = H_fd(n0, h, case_map)
        rel = fro_norm(Hfd - Ha) / max(DT("1.0"), fro_norm(Ha))
        print(f"h={h:g}  rel_err={rel:.3e}")

M = 18
case_map = {'AB_XX': 'else', 'AB_YY': 'else'}
First 12 vars: ['AA_XX', 'AA_XY', 'AA_YY', 'AB_XX', 'AB_XY', 'AB_YY', 'AC_XX', 'AC_XY', 'AC_YY', 'BB_XX', 'BB_XY', 'BB_YY'] ...

G_ex(n0) = 4510.0657675975595877
||H_analytic||_F = 95.15030260585858
analytic symmetry err = 0.0
h=0.01  rel_err=6.108e-06
h=0.001  rel_err=6.108e-08
h=0.0001  rel_err=2.513e-09
h=1e-05  rel_err=2.452e-07
h=1e-06  rel_err=2.508e-05
